# មនុស្សនៅក្នុងខ្សែ: ទ្វារមុនសកម្មភាព ការរៀបចំហានិភ័យ និងកំណត់ហេតុត្រួតពិនិត្យ

README សម្រាប់មេរៀននេះណែនាំមនុស្សនៅក្នុងខ្សែជាមួយការចេញផ្សាយខ្លីមួយដែលសួរឱ្យអ្នកប្រើប្រាស់ `APPROVE` ឬ `REJECT` បន្ទាប់ពីភ្នាក់ងារបានបង្កើតចម្លើយមួយរួច។ គំរូនោះជាចំណុចចាប់ផ្តើមល្អ ប៉ុន្តែការអនុវត្តเงินจริงនៃ HITL ជាទូទៅត្រូវការផ្នែកបន្ថែមបីនេះ៖

1. **ទ្វារមុនសកម្មភាព** ដែលដំណើរការមុនឱ្យភ្នាក់ងារធ្វើជំហានដែលមានហានិភ័យ ដើម្បីការគ្រប់គ្រងថ្លៃដើម អត្រាការមិនអាចត្រឡប់ក្រោយ និងពេលយឺត។
2. **ការរៀបចំកម្រិតហានិភ័យ** ដូច្នេះសកម្មភាពដែលមានហានិភ័យតិចអាចដំណើរការបានដោយស្វ័យប្រវត្តិ សកម្មភាពដែលមានហានិភ័យមធ្យមត្រូវបានអនុម័តជាក្រុម ហើយមានតែសកម្មភាពដែលមានហានិភ័យខ្ពស់ប៉ុណ្ណោះដែលរាំងខ្ទប់លើមនុស្ស។
3. កំណត់ហេតុត្រួតពិនិត្យ និងលក្ខខណ្ឌកែប្រែ ដូច្នេះការសម្រេចចិត្តរបស់ទ្វារ នីតិវិធីទាំងឡាយត្រូវបានកត់ត្រាជា JSONL ហើយការបដិសេធនាំឱ្យភ្នាក់ងារត្រូវបានវាយតម្លៃឡើងវិញជាមួយមូលហេតុដែលមានរចនាសម្ព័ន្ធជំនួសការបោះពុម្ពប្រាប់ថា `Revising...`។

សៀវភៅចំណាំនេះសាងសង់តាមលើមូលដ្ឋានដូចគ្នានឹង `06-system-message-framework.ipynb`។ វាដំណើរការចាប់ពីដើមបញ្ចប់ក្នុង `DEMO_MODE = True` (មិនបាច់បញ្ចូលអន្តរកម្ម) ឬជាមួយការបញ្ចូលពិតប្រាកដដោយ `input()` ពេល `DEMO_MODE = False`។ ចំណាំៈ ក្នុង DEMO_MODE ការបោះព្យាយាមលើគោលបំណងទីបីត្រូវបានសរសេរជាស្ក្រីប ដូច្នេះគ្រឿងចក្រ​លូបត្រូវឃើញច្បាស់ពីដើមដល់ចប់។ ការកំណត់ថ្នាក់ឡើងវិញដោយការបង្រៀនពិតតម្រូវឱ្យ `DEMO_MODE = False` និងអ្នកប្រតិបត្តការ។

**មិនសមរម្យសម្រាប់វឌ្ឍនភាព (បានគ្រប់គ្រងនៅមេរៀនផ្សេងទៀត):** ការផ្ទៀងផ្ទាត់ និងការគ្រប់គ្រងការចូលប្រើ (ការគំរាមកំហែងនៅ Lesson 06 README), មេឌៀកណ្សា ខ្សែបញ្ជាឧបករណ៍ (Lesson 14 MAF ផ្នែកជ្រៅ), ការប្រកួតប្រជែងរវាងភ្នាក់ងារច្រើន។

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## លំនាំទី 1៖ ទ្វារមុនសកម្មភាព

ខ្លឹមសារដែលនៅក្នុង README HITL គឺហៅភ្នាក់ងារមុន បន្ទាប់មកស្នើឲ្យអ្នកប្រើអនុម័តលទ្ធផល។ នេះគឺជាប្រតិបត្តិការប្រកាសដោយ **បន្ទាប់សកម្មភាព**។ ភ្នាក់ងារបានអនុវត្តរួចហើយ ដូច្នេះការហៅ LLM ត្រូវបានបង់ប្រាក់រួចហើយ ហើយផលបរិយាកាសណាមួយ (ខ័ណ្ឌអ៊ីមែលដែលបានផ្ញើ ចំណុចទិន្នន័យបញ្ជូលជាថ្មី រឺមតិយោបល់ដែលបានបង្ហោះ) បានកើតមានរួចហើយ។

ប្រតិបត្តិការដោយ **មុនសកម្មភាព** ជាទ្វារដាក់តាំងមុនពេលភ្នាក់ងារប្រតិបត្តិជំហានដែលមានហានិភ័យ។ ភ្នាក់ងារបញ្ជូនសេចក្តីស្នើ សភាពទ្វារសម្រេចថាតើត្រូវអនុវត្ត ឬអត់ ហើយមានតែនៅពេលអនុម័តនៅពេលហែតុផលកើតមាន។

| សមាសភាព | សេចក្តីអនុម័តបន្ទាប់សកម្មភាព (ខ្លឹមសារ README) | ទ្វាមុនសកម្មភាព (សៀវភៅកំណត់ត្រានេះ) |
|---|---|---|
| ពេលណាដែលការអនុម័តដំណើរការ? | បន្ទាប់ពីភ្នាក់ងារបង្កើតលទ្ធផល | មុនពេលផលបរិយាកាសណាមួយប្រតិបត្តិការ |
| ការចំណាយ LLM នៅពេលបដិសេធ | បានបង់រួចហើយ | បង់តែសម្រាប់សំណើមិនមែនសម្រាប់សកម្មភាព |
| ផលបរិយាកាសដែលមិនអាចបម្លែងវិញ | អាចកើតមាន (សកម្មភាពបានកើតឡើងរួចហើយ) | ត្រូវបានទប់ស្កាត់ |
| ភាពច្បាស់លាស់ក្នុងការត្រួតពិនិត្យ | ការអនុម័តជាពាក្យបង្ហាញ | ការអនុម័តជាកំណត់ត្រា JSON មានពេលវេលា សកម្មភាព ហេតុផល |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## លំនាំទី 2៖ កំណត់កម្រិតហានិភ័យ

មិនមែនសកម្មភាពទាំងអស់ទាមទារការអនុម័តពីមនុស្សទេ។ ការស្វែងរកត្រឹមតែអានតាម API សាធារណៈមានហានិភ័យខុសពីការផ្ញើអ៊ីមែលទៅអតិថិជន។ ការពិចារណាទាំងពីរដូចគ្នាធ្វើឱ្យអ្នកប្រតិបត្តិអត់ចំណាប់អារម្មណ៍ ហើយធ្វើឱ្យភ្នាក់ងារយឺត។

ម៉ូដែល 3 កម្រិតសាមញ្ញ៖

| កម្រិត | ឧទាហរណ៍ | ដំណើរការ​អនុម័ត |
|---|---|---|
| `ទាប` (អានតែ) | ស្វែងរកមូលដ្ឋានចំណេះដឹង ស្វែងរកជម្រើស​ទេសចរណ៍ ទាញយក​ទំព័របណ្ដាញសាធារណៈ | រត់ស្វ័យប្រវត្តិ ចុះបញ្ជីសម្រាប់ពិនិត្យឡើងវិញ |
| `មធ្យម` (កែប្រែថ្លៃសេវាបិទ) | បញ្ចូនលទ្ធផលរក្សាទុក បន្ថែមគណនី កំណត់កាលបរិច្ឆេទរំលឹក | រត់ស្វ័យប្រវត្តិ ប៉ុន្តែពិនិត្យជា​កញ្ចប់រៀងរាល់ថ្ងៃ |
| `ខ្ពស់` (មុខងារចូលទៅក្រៅ ឬ មិនអាចបញ្ចប់) | ផ្ញើអ៊ីមែល ប្រាក់គិតថ្លៃ ការបញ្ចេញសារទៅឆានែលសាធារណៈ | បិទរហូតដល់មានការអនុម័តពីមនុស្ស |

នេះគឺជាកម្រិតមួយ។ ប្រព័ន្ធផលិតកម្មជាញឹកញាប់ប្រើកម្រិតរមောင်းច្រើនជាងនេះ (ឧ. កម្រិតអាជ្ញាបណ្ណ AWS IAM, កម្រិតចូលដំណើរការតាមតួនាទី)។ រូបមន្ត 3 កម្រិតខាងក្រោមគឺជាកំណែតូចជាងគេដែលមានប្រយោជន៍សម្រាប់ភ្នាក់ងារដែលបញ្ចូលសកម្មភាពអានតែ និងមានផលប៉ៈពាល់។

កម្មវិធីចាត់ថ្នាក់ខាងក្រោមប្រើក្បួនគន្លឹះពាក្យដើម្បីឲ្យការបង្ហាញត្រូវបានកំណត់ និងថ្លៃសេវាប្រាក់តិច។ នៅក្នុងប្រព័ន្ធផលិតកម្ម អ្នកនឹងប្តូរវានៅជំនួសដោយកម្មវិធីចាត់ថ្នាក់ដែលបានរៀន ឬម៉ាស៊ីនគោលនយោបាយ។

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## លំនាំ 3: កំណត់ហេតុអូឌីត និងរង្វិលការពិនិត្យឡើងវិញ

`print("Response approved.")` មិនមែនជាកំណត់ហេតុអូឌីតទេ។ សម្រាប់ការជឿទុកចិត្ត ការសម្រេចចិត្តរាល់ទ្វារត្រូវតែត្រូវបានកត់ត្រាជាអំពើដែលមានរចនាសម្ព័ន្ធ ដែលអ្នកអាចស្វែងរក បញ្ចេញឡើងវិញ ឬភ្ជាប់ទៅនឹងការពិនិត្យហេតុការណ៍មួយបានក្រោយមក។

មានពីរដំណើរការ៖

1. **JSONL បន្ថែមតែប៉ុណ្ណោះ។** មួយបន្ទាត់សម្រាប់ការសម្រេចចិត្តមួយ មានពេលវេលា សកម្មភាព ជួរ សម្រេចចិត្ត និងហេតុផល។ ងាយស្រួលស្វែងរក និងងាយស្រួលផ្ញើទៅហាងកំណត់ហេតុពិតប្រាកដក្រោយ។
2. **រង្វិលការពិនិត្យឡើងវិញពេលបដិសេធ។** ពេលទ្វារបញ្ជូន `deny` អ្នកចាត់ការទម្លាក់សារ​ដាក់ខ្លួនឯងជាមួយហេតុផលបដិសេធក្នុងcontext ដូច្នេះការផ្តល់យោបល់បន្ទាប់អាចជៀសវាងបញ្ហា។

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## ធនធានបន្ថែម

គម្រោងសាធារណៈផ្សេងទៀតជាច្រើនអនុវត្តលំនាំ HITL ទាំងនេះជាការបម្លែង។ ប្រៀបធៀបវិធីសាស្ត្រដើម្បីស្វែងរកអ្វីដែលសមនឹងស៊្តាក់របស់អ្នក៖

- **LangChain** ការការវេចខ្ចប់ឧបករណ៍ human-in-the-loop ([ឯកសារ](https://python.langchain.com/docs/integrations/tools/human_tools)): ការវេចខ្ចប់ឧបករណ៍ដែលអាចជម្រះបានដែលបញ្ឈប់ការអនុវត្តសម្រាប់ការបញ្ចូលពីមនុស្ស។
- **AutoGen** `UserProxyAgent` ([ឯកសារ v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ បានផ្លាស់ប្តូរនេះ): ប្រើតួអង្គភេរវកម្មជាពិសេសដើម្បីតំណាងឲ្យមនុស្សក្នុងការសន្ទនាច្រើនភេរវកម្ម។
- **Semantic Kernel** ការត្រង់មុខងារ ([ឯកសារ](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): មេឌៀវែរដែលដំណើរការឥឡូវនេះជុំវិញការហៅមុខងារ រាល់មុខងារ សមស្របសម្រាប់តម្រូវការសម្រួលលក្ខខណ្ឌ។

គម្រោងនីមួយៗដោះស្រាយលំនាំរងទាំងបីយ៉ាងខុសៗគ្នា៖ LangChain វេចខ្ចប់ពួកវាជាឧបករណ៍, AutoGen ប្រើតួនាទីភេរវកម្ម, Semantic Kernel ប្រើត្រង់មេឌៀវែរ។ អានការអភិវឌ្ឍមួយឬពីរពេញលេញមុនពេលជ្រើសរើសការរចនាសម្រាប់ភេរវកម្មរបស់អ្នក។

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
